In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")

# Q1 – K-Fold Cross Validation for Multiple Linear Regression

In [ ]:
DATA_PATH = None

def load_house_dataset():
    if DATA_PATH is not None and Path(DATA_PATH).exists():
        return pd.read_csv(DATA_PATH)

    candidates = [
        "USA_Housing.csv",
        "USA_Housing_Data.csv",
        "usa_housing.csv"
    ]
    for filename in candidates:
        if Path(filename).exists():
            print(f"Using local file: {filename}")
            return pd.read_csv(filename)

    print("No USA House Price CSV was found automatically.")
    print("Please upload the CSV in your notebook environment and run this cell again,")
    print("or set DATA_PATH to the complete CSV path.")
    return None

house_df = load_house_dataset()

if house_df is not None:
    print("Dataset shape:", house_df.shape)
    display(house_df.head())
    print("\nColumns:")
    print(house_df.columns.tolist())

In [ ]:
if house_df is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            uploaded_name = next(iter(uploaded))
            house_df = pd.read_csv(uploaded_name)
            print("Loaded:", uploaded_name)
            print("Dataset shape:", house_df.shape)
    except ImportError:
        print("Google Colab upload is unavailable in this environment.")

In [ ]:
assert house_df is not None, "Load the USA Housing CSV first."

price_candidates = [c for c in house_df.columns if c.strip().lower() == "price"]

if not price_candidates:

    price_candidates = [c for c in house_df.columns if "price" in c.lower()]

if not price_candidates:
    raise ValueError("Could not find the price column. Set the target column manually.")

TARGET = price_candidates[0]
print("Target column:", TARGET)

X_raw = house_df.drop(columns=[TARGET]).copy()
y = pd.to_numeric(house_df[TARGET], errors="coerce")

valid_target = y.notna()
X_raw = X_raw.loc[valid_target].reset_index(drop=True)
y = y.loc[valid_target].reset_index(drop=True)

print("X shape:", X_raw.shape)
print("y shape:", y.shape)

display(X_raw.head())

In [ ]:
numeric_features = X_raw.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_raw.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ],
    remainder="drop"
)

X = preprocessor.fit_transform(X_raw)
feature_names = preprocessor.get_feature_names_out()

print("Original feature count:", X_raw.shape[1])
print("Features after preprocessing:", X.shape[1])
print("First few transformed features:", feature_names[:10])

In [ ]:
def least_squares_beta(X_train, y_train):
    """Return beta for y = beta_0 + beta_1*x1 + ... using the pseudoinverse."""
    X_aug = np.column_stack([np.ones(X_train.shape[0]), X_train])
    beta = np.linalg.pinv(X_aug) @ y_train
    return beta

def predict_with_beta(X_data, beta):
    X_aug = np.column_stack([np.ones(X_data.shape[0]), X_data])
    return X_aug @ beta

print("Least-squares functions defined.")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_results = []
betas = []

for fold_number, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()

    beta = least_squares_beta(X_train, y_train)
    y_pred = predict_with_beta(X_test, beta)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    betas.append(beta)
    fold_results.append({
        "Fold": fold_number,
        "Training samples": len(train_idx),
        "Test samples": len(test_idx),
        "R2_score": r2,
        "RMSE": rmse
    })

fold_results_df = pd.DataFrame(fold_results)
display(fold_results_df)

best_row = fold_results_df.loc[fold_results_df["R2_score"].idxmax()]
best_fold = int(best_row["Fold"])
best_beta = betas[best_fold - 1]

print(f"Best fold = {best_fold}")
print(f"Maximum validation R2 = {best_row['R2_score']:.6f}")
print("Best beta shape:", best_beta.shape)

In [ ]:
beta_table = pd.DataFrame({
    "Feature": ["Intercept"] + list(feature_names),
    "Beta": best_beta
})

display(beta_table.head(20))

print("Number of coefficients:", len(beta_table))

In [ ]:
X_train70, X_test30, y_train70, y_test30 = train_test_split(
    X, y.to_numpy(),
    test_size=0.30,
    random_state=RANDOM_STATE
)

beta_70 = least_squares_beta(X_train70, y_train70)
y_pred30 = predict_with_beta(X_test30, beta_70)

r2_70_30 = r2_score(y_test30, y_pred30)
rmse_70_30 = np.sqrt(mean_squared_error(y_test30, y_pred30))
mae_70_30 = mean_absolute_error(y_test30, y_pred30)

print("Q1 – Final 70/30 results")
print(f"R2 score : {r2_70_30:.6f}")
print(f"RMSE     : {rmse_70_30:,.2f}")
print(f"MAE      : {mae_70_30:,.2f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test30, y_pred30, alpha=0.6)
min_val = min(y_test30.min(), y_pred30.min())
max_val = max(y_test30.max(), y_pred30.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Q1: Actual vs Predicted House Price")
plt.tight_layout()
plt.show()

# Q2 – Multiple Linear Regression using Gradient Descent

In [ ]:
X_train_q2, X_temp_q2, y_train_q2, y_temp_q2 = train_test_split(
    X, y.to_numpy(),
    test_size=0.44,
    random_state=RANDOM_STATE
)

X_val_q2, X_test_q2, y_val_q2, y_test_q2 = train_test_split(
    X_temp_q2, y_temp_q2,
    test_size=(30/44),
    random_state=RANDOM_STATE
)

print("Training :", X_train_q2.shape[0], f"({X_train_q2.shape[0]/len(X)*100:.2f}%)")
print("Validation:", X_val_q2.shape[0], f"({X_val_q2.shape[0]/len(X)*100:.2f}%)")
print("Testing   :", X_test_q2.shape[0], f"({X_test_q2.shape[0]/len(X)*100:.2f}%)")

In [ ]:
def gradient_descent_linear_regression(X_train, y_train, learning_rate, iterations=100):
    X_aug = np.column_stack([np.ones(X_train.shape[0]), X_train])
    beta = np.zeros(X_aug.shape[1], dtype=float)

    cost_history = []

    m = len(y_train)

    for _ in range(iterations):
        predictions = X_aug @ beta
        errors = predictions - y_train

        gradient = (X_aug.T @ errors) / m
        beta -= learning_rate * gradient

        cost = np.mean(errors ** 2) / 2
        cost_history.append(cost)

    return beta, cost_history

def gd_predict(X_data, beta):
    X_aug = np.column_stack([np.ones(X_data.shape[0]), X_data])
    return X_aug @ beta

learning_rates = [0.001, 0.01, 0.1, 1.0]
gd_results = {}
rows = []

for lr in learning_rates:
    beta_lr, cost_history = gradient_descent_linear_regression(
        X_train_q2, y_train_q2, learning_rate=lr, iterations=100
    )

    val_pred = gd_predict(X_val_q2, beta_lr)
    test_pred = gd_predict(X_test_q2, beta_lr)

    val_r2 = r2_score(y_val_q2, val_pred)
    test_r2 = r2_score(y_test_q2, test_pred)

    gd_results[lr] = {
        "beta": beta_lr,
        "cost": cost_history,
        "val_pred": val_pred,
        "test_pred": test_pred
    }

    rows.append({
        "Learning Rate": lr,
        "Validation R2": val_r2,
        "Test R2": test_r2,
        "Final Training Cost": cost_history[-1]
    })

gd_results_df = pd.DataFrame(rows)
display(gd_results_df)

In [ ]:
best_lr_row = gd_results_df.loc[gd_results_df["Validation R2"].idxmax()]
best_lr = float(best_lr_row["Learning Rate"])

print(f"Best learning rate = {best_lr}")
print(f"Best validation R2 = {best_lr_row['Validation R2']:.6f}")
print(f"Corresponding test R2 = {best_lr_row['Test R2']:.6f}")

In [ ]:
plt.figure(figsize=(8, 5))

for lr in learning_rates:
    plt.plot(range(1, 101), gd_results[lr]["cost"], label=f"α = {lr}")

plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Q2: Gradient Descent Cost vs Iteration")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
best_test_pred = gd_results[best_lr]["test_pred"]

plt.figure(figsize=(7, 5))
plt.scatter(y_test_q2, best_test_pred, alpha=0.6)
min_val = min(y_test_q2.min(), best_test_pred.min())
max_val = max(y_test_q2.max(), best_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Q2: Best Gradient Descent Model (α={best_lr})")
plt.tight_layout()
plt.show()

# Q2 – Explanation of the result

# Q3 – Pre-processing and Multiple Linear Regression

In [ ]:
Q3_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"

columns = [
    "symboling", "normalized_losses", "make", "fuel_type", "aspiration",
    "num_doors", "body_style", "drive_wheels", "engine_location",
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_type", "num_cylinders", "engine_size", "fuel_system",
    "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
    "city_mpg", "highway_mpg", "price"
]

try:
    auto_df = pd.read_csv(
        Q3_URL,
        header=None,
        names=columns,
        na_values="?"
    )
except Exception as e:
    print("Could not download automatically:", e)
    print("Download imports-85.data manually and set Q3_PATH below.")

print("Q3 dataset shape:", auto_df.shape)
display(auto_df.head())

In [ ]:
missing_before = auto_df.isna().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0])

print("Rows:", len(auto_df))
print("Columns:", len(auto_df.columns))

In [ ]:
numeric_expected = [
    "symboling", "normalized_losses", "wheel_base", "length", "width",
    "height", "curb_weight", "engine_size", "bore", "stroke",
    "compression_ratio", "horsepower", "peak_rpm", "city_mpg",
    "highway_mpg", "price"
]

for col in numeric_expected:
    auto_df[col] = pd.to_numeric(auto_df[col], errors="coerce")

for col in auto_df.columns:
    if col == "price":
        continue

    if pd.api.types.is_numeric_dtype(auto_df[col]):
        auto_df[col] = auto_df[col].fillna(auto_df[col].median())
    else:
        mode = auto_df[col].mode(dropna=True)
        if not mode.empty:
            auto_df[col] = auto_df[col].fillna(mode.iloc[0])

auto_df = auto_df.dropna(subset=["price"]).reset_index(drop=True)

print("Remaining rows after preprocessing missing values:", len(auto_df))
print("Missing values remaining:")
display(auto_df.isna().sum().sort_values(ascending=False).head(10))

# Q3 categorical conversion

In [ ]:
number_words = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
    "ten": 10, "eleven": 11, "twelve": 12
}

for col in ["num_doors", "num_cylinders"]:
    auto_df[col] = auto_df[col].astype(str).str.lower().map(number_words)

for col in ["num_doors", "num_cylinders"]:
    auto_df[col] = pd.to_numeric(auto_df[col], errors="coerce")
    auto_df[col] = auto_df[col].fillna(auto_df[col].median())

auto_df = pd.get_dummies(
    auto_df,
    columns=["body_style", "drive_wheels"],
    drop_first=False,
    dtype=int
)

label_columns = ["make", "aspiration", "engine_location", "fuel_type"]

label_encoders = {}

for col in label_columns:
    le = LabelEncoder()
    auto_df[col] = le.fit_transform(auto_df[col].astype(str))
    label_encoders[col] = le

auto_df["fuel_system"] = (
    auto_df["fuel_system"].astype(str).str.lower().str.contains("pfi")
).astype(int)

auto_df["engine_type"] = (
    auto_df["engine_type"].astype(str).str.lower().str.contains("ohc")
).astype(int)

print("Encoding completed.")
display(auto_df.head())
print("\nData types:")
display(auto_df.dtypes)

In [ ]:
for col in auto_df.columns:
    if col == "price":
        continue
    if auto_df[col].isna().any():
        auto_df[col] = auto_df[col].fillna(auto_df[col].median())

print("Total missing values:", auto_df.isna().sum().sum())
print("Final shape:", auto_df.shape)

In [ ]:
X_auto = auto_df.drop(columns=["price"]).astype(float)
y_auto = auto_df["price"].astype(float)

X_auto_train, X_auto_test, y_auto_train, y_auto_test = train_test_split(
    X_auto, y_auto,
    test_size=0.30,
    random_state=RANDOM_STATE
)

scaler_auto = StandardScaler()
X_auto_train_scaled = scaler_auto.fit_transform(X_auto_train)
X_auto_test_scaled = scaler_auto.transform(X_auto_test)

print("Training samples:", len(X_auto_train))
print("Testing samples :", len(X_auto_test))
print("Number of input features:", X_auto_train_scaled.shape[1])

In [ ]:
linear_model_auto = LinearRegression()
linear_model_auto.fit(X_auto_train_scaled, y_auto_train)

y_auto_pred = linear_model_auto.predict(X_auto_test_scaled)

q3_r2 = r2_score(y_auto_test, y_auto_pred)
q3_rmse = np.sqrt(mean_squared_error(y_auto_test, y_auto_pred))
q3_mae = mean_absolute_error(y_auto_test, y_auto_pred)

print("Q3 – Linear Regression")
print(f"R2 score : {q3_r2:.6f}")
print(f"RMSE     : {q3_rmse:,.2f}")
print(f"MAE      : {q3_mae:,.2f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_auto_test, y_auto_pred, alpha=0.7)
min_val = min(y_auto_test.min(), y_auto_pred.min())
max_val = max(y_auto_test.max(), y_auto_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Q3: Linear Regression – Actual vs Predicted")
plt.tight_layout()
plt.show()

# Q3 Step 6 – PCA + Linear Regression

In [ ]:
pca_full = PCA(n_components=0.95, svd_solver="full")
X_auto_train_pca = pca_full.fit_transform(X_auto_train_scaled)
X_auto_test_pca = pca_full.transform(X_auto_test_scaled)

print("Original number of features:", X_auto_train_scaled.shape[1])
print("Number of PCA components:", X_auto_train_pca.shape[1])
print("Explained variance retained:",
      f"{pca_full.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
pca_model = LinearRegression()
pca_model.fit(X_auto_train_pca, y_auto_train)

y_pca_pred = pca_model.predict(X_auto_test_pca)

pca_r2 = r2_score(y_auto_test, y_pca_pred)
pca_rmse = np.sqrt(mean_squared_error(y_auto_test, y_pca_pred))
pca_mae = mean_absolute_error(y_auto_test, y_pca_pred)

print("Q3 – PCA + Linear Regression")
print(f"R2 score : {pca_r2:.6f}")
print(f"RMSE     : {pca_rmse:,.2f}")
print(f"MAE      : {pca_mae:,.2f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Multiple Linear Regression",
        "PCA + Multiple Linear Regression"
    ],
    "R2": [q3_r2, pca_r2],
    "RMSE": [q3_rmse, pca_rmse],
    "MAE": [q3_mae, pca_mae]
})

display(comparison)

if pca_r2 > q3_r2:
    print("PCA model has the higher test R2 on this split.")
elif pca_r2 < q3_r2:
    print("Original feature model has the higher test R2 on this split.")
else:
    print("Both models have the same test R2 on this split.")

In [ ]:
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.95, linestyle="--", label="95% variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Q3: PCA Explained Variance")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Q1
# Q2
# Q3